# Section 6.7 — Real study: operational and conduct endpoints

Reads `data/study_percall_deidentified.csv` (fifteen models A–O, fifteen vignettes, two administration conditions; model identities and vignette text withheld) and reproduces Figure 7 and Table 6.5.

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
df = pd.read_csv("../data/study_percall_deidentified.csv")
api = df[df.condition.str.startswith("API")].copy(); chat = df[df.condition.str.startswith("Chat")].copy()
for c in ["output_tokens", "reasoning_tokens", "cost_upstream_usd", "answer_chars"]:
    api[c] = pd.to_numeric(api[c], errors="coerce")
chat["answer_chars"] = pd.to_numeric(chat["answer_chars"], errors="coerce")
summary = api.groupby("model").agg(n=("item", "size"), out_med=("output_tokens", "median"),
    reasoning_share=("reasoning_tokens", lambda s: np.nan if s.sum() == 0 else 100*s.sum()/api.loc[s.index, "output_tokens"].sum()),
    cost_mean=("cost_upstream_usd", "mean"), len_med=("answer_chars", "median"), providers=("n_providers_for_model", "max"))
print(summary.round(4))

In [ ]:
# paired condition effect on answer length, chat minus API, Holm-adjusted
rows = []
for m, g in chat.groupby("model"):
    a = api[api.model == m].set_index("item")["answer_chars"]; c = g.set_index("item")["answer_chars"]
    items = a.index.intersection(c.index); d = (c[items] - a[items]).astype(float)
    rows.append(dict(model=m, n=len(items), mean=d.mean(), se=d.std(ddof=1)/np.sqrt(len(d)), p=stats.ttest_1samp(d, 0).pvalue))
pd_ = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
m = len(pd_); pd_["holm"] = [min(1, (m - i)*p) for i, p in enumerate(pd_.p)]; pd_["holm"] = pd_["holm"].cummax()
print(pd_.sort_values("model").round(4))

In [ ]:
# shared item effect: pairwise correlation of log output tokens and two-way decomposition
L = api.assign(l=np.log(api.output_tokens)).pivot(index="model", columns="item", values="l")
cors = [L.loc[a].corr(L.loc[b]) for i, a in enumerate(L.index) for b in L.index[i+1:]]
print("mean pairwise r", np.nanmean(cors).round(2))
Lc = L.dropna(axis=1); gm = Lc.values.mean(); mi = Lc.mean(0); mm = Lc.mean(1)
ss_item = Lc.shape[0]*((mi-gm)**2).sum(); ss_model = Lc.shape[1]*((mm-gm)**2).sum()
ss_res = ((Lc.sub(mm, axis=0).sub(mi, axis=1) + gm)**2).values.sum()
tot = ss_item + ss_model + ss_res
print(f"item {100*ss_item/tot:.0f}%  model {100*ss_model/tot:.0f}%  residual {100*ss_res/tot:.0f}%")